# 01 — Data Cleaning
## Patient No-Show & Clinic Efficiency Analysis

**Objective of this notebook:** take the raw Kaggle "Medical Appointment No Shows" dataset and turn it into a
clean, analysis-ready dataset — fixing structural issues, handling invalid records, and engineering a small set
of business-relevant features. No exploratory analysis or charts happen here — that's Notebook 02.

Real-world datasets typically have structural issues — typos in column names, invalid values, inconsistent types — that need to be resolved before any analysis is reliable.

**Grain confirmed in Phase 1:** one row = one scheduled appointment (`AppointmentID` is unique; `PatientId` repeats).


In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

RAW_PATH = '../data/raw/KaggleV2-May-2016.csv'
df = pd.read_csv(RAW_PATH)

print("Shape:", df.shape)
df.head()


Shape: (110527, 14)


,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show
0,2.987250e+13,5642903,F,2016-04-29T18:38:08Z,2016-04-29T00:00:00Z,62,JARDIM DA PENHA,0,1,0,0,0,0,No
1,5.589978e+14,5642503,M,2016-04-29T16:08:27Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,0,0,0,0,0,No
2,4.262962e+12,5642549,F,2016-04-29T16:19:04Z,2016-04-29T00:00:00Z,62,MATA DA PRAIA,0,0,0,0,0,0,No
3,8.679512e+11,5642828,F,2016-04-29T17:29:31Z,2016-04-29T00:00:00Z,8,PONTAL DE CAMBURI,0,0,0,0,0,0,No
4,8.841186e+12,5642494,F,2016-04-29T16:07:23Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,1,1,0,0,0,No


## 1. Structural Cleaning

### 1.1 Standardize column names
The source file has inconsistent naming and two typos (`Hipertension`, `Handcap`). We standardize to clear,
consistent `snake_case` names so later SQL/Python/Power BI work references stable, readable column names.


In [2]:
rename_map = {
    'PatientId': 'patient_id',
    'AppointmentID': 'appointment_id',
    'Gender': 'gender',
    'ScheduledDay': 'scheduled_day',
    'AppointmentDay': 'appointment_day',
    'Age': 'age',
    'Neighbourhood': 'neighbourhood',
    'Scholarship': 'scholarship',
    'Hipertension': 'hypertension',   # fixing source typo
    'Diabetes': 'diabetes',
    'Alcoholism': 'alcoholism',
    'Handcap': 'handicap',            # fixing source typo
    'SMS_received': 'sms_received',
    'No-show': 'no_show'
}
df = df.rename(columns=rename_map)
df.columns.tolist()


['patient_id',
 'appointment_id',
 'gender',
 'scheduled_day',
 'appointment_day',
 'age',
 'neighbourhood',
 'scholarship',
 'hypertension',
 'diabetes',
 'alcoholism',
 'handicap',
 'sms_received',
 'no_show']

### 1.2 Convert data types
`scheduled_day` and `appointment_day` are stored as text. We convert both to real datetimes so we can do date
arithmetic (lead time, weekday, month) later. `patient_id` is stored as a float in the source file purely due to
its size — we convert it to a clean integer-like string ID since we never do arithmetic on it, only grouping.


In [3]:
df['scheduled_day'] = pd.to_datetime(df['scheduled_day'])
df['appointment_day'] = pd.to_datetime(df['appointment_day'])

# patient_id is an identifier, not a quantity -> keep as a stable integer-backed string
df['patient_id'] = df['patient_id'].astype('int64').astype(str)

df.dtypes


patient_id                         str
appointment_id                   int64
gender                             str
scheduled_day      datetime64[us, UTC]
appointment_day    datetime64[us, UTC]
age                              int64
neighbourhood                      str
scholarship                      int64
hypertension                     int64
diabetes                         int64
alcoholism                       int64
handicap                         int64
sms_received                     int64
no_show                            str
dtype: object

### 1.3 Duplicate check
Confirmed in Phase 1 (and again on the Excel `Clean_Data` audit tab) that there are zero fully duplicate rows and
zero duplicate `appointment_id` values. We re-verify here rather than assuming Phase 1's finding still holds.


In [4]:
print("Fully duplicate rows:", df.duplicated().sum())
print("Duplicate appointment_id:", df['appointment_id'].duplicated().sum())


Fully duplicate rows: 0
Duplicate appointment_id: 0


### 1.4 Missing values
Phase 1 found zero missing values across all columns. Re-verified below.


In [5]:
df.isnull().sum()


patient_id         0
appointment_id     0
gender             0
scheduled_day      0
appointment_day    0
age                0
neighbourhood      0
scholarship        0
hypertension       0
diabetes           0
alcoholism         0
handicap           0
sms_received       0
no_show            0
dtype: int64

### 1.5 Invalid `age` values
Phase 1 found exactly **one** record with `age = -1`, which is not a valid age. We drop this single record rather
than guessing a replacement value, since guessing would violate the "never invent results" rule and one row has
no meaningful effect on 110K+ records.


In [6]:
invalid_age = df[df['age'] < 0]
print("Invalid age records found:", len(invalid_age))
invalid_age


Invalid age records found: 1


,patient_id,appointment_id,gender,scheduled_day,appointment_day,age,neighbourhood,scholarship,hypertension,diabetes,alcoholism,handicap,sms_received,no_show
99832,465943158731293,5775010,F,2016-06-06 08:58:13+00:00,2016-06-06 00:00:00+00:00,-1,ROMÃO,0,0,0,0,0,0,No


In [7]:
before = len(df)
df = df[df['age'] >= 0].copy()
after = len(df)
print(f"Dropped {before - after} row(s) with invalid age. Rows remaining: {after}")


Dropped 1 row(s) with invalid age. Rows remaining: 110526


### 1.6 Invalid lead time (scheduled after the appointment date)
Lead time = days between `scheduled_day` and `appointment_day`. Phase 1 found **5 records** where this is negative
— i.e. the appointment was logged as scheduled *after* the appointment date, which is not logically possible and
is almost certainly a data-entry error. We compute lead time now (so we can find and drop them), and recompute it
cleanly afterward as an engineered feature in Section 2.


In [8]:
lead_time_check = (df['appointment_day'].dt.normalize() - df['scheduled_day'].dt.normalize()).dt.days
invalid_lead = df[lead_time_check < 0]
print("Invalid (negative) lead time records found:", len(invalid_lead))
invalid_lead[['patient_id','appointment_id','scheduled_day','appointment_day']]


Invalid (negative) lead time records found: 5


,patient_id,appointment_id,scheduled_day,appointment_day
27033,7839272661752,5679978,2016-05-10 10:51:53+00:00,2016-05-09 00:00:00+00:00
55226,7896293967868,5715660,2016-05-18 14:50:41+00:00,2016-05-17 00:00:00+00:00
64175,24252258389979,5664962,2016-05-05 13:43:58+00:00,2016-05-04 00:00:00+00:00
71533,998231581612122,5686628,2016-05-11 13:49:20+00:00,2016-05-05 00:00:00+00:00
72362,3787481966821,5655637,2016-05-04 06:50:57+00:00,2016-05-03 00:00:00+00:00


In [9]:
before = len(df)
df = df[lead_time_check >= 0].copy()
after = len(df)
print(f"Dropped {before - after} row(s) with negative lead time. Rows remaining: {after}")


Dropped 5 row(s) with negative lead time. Rows remaining: 110521


### 1.7 Standardize categorical fields
`gender` (F/M) and `no_show` (Yes/No) are already clean, consistent categories — no fuzzy variants like "female"
or "y/n" were found, so no further standardization is needed here. We do rename `no_show` values into an explicit
binary flag below (Section 2) since "No-show = Yes" reads as confusing/counter-intuitive.

`handicap` is **not actually binary** — it ranges 0–4 (likely a degree/count of disability). We leave its raw
values intact and do not force it into a 0/1 flag, since that would discard real information the dataset provides.


In [10]:
print("gender values:", df['gender'].unique())
print("no_show values:", df['no_show'].unique())
print("handicap values:", sorted(df['handicap'].unique()))


gender values: <StringArray>
['F', 'M']
Length: 2, dtype: str
no_show values: <StringArray>
['No', 'Yes']
Length: 2, dtype: str
handicap values: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]


## 2. Feature Engineering

Every feature below is created because it answers a specific business question from Phase 0 — we don't add
columns just to add them.


In [11]:
# no_show_flag: binary target, 1 = patient did NOT show up. Needed because "No-show"='Yes' is confusing to read
# in aggregations (SUM/MEAN work naturally on 0/1, not on 'Yes'/'No' strings).
df['no_show_flag'] = (df['no_show'] == 'Yes').astype(int)

# lead_time_days: days between booking and appointment. This is one of the strongest observed drivers
# of no-shows (see Phase 2 Excel audit: 4.6% same-day vs ~33% at 31+ days).
df['lead_time_days'] = (df['appointment_day'].dt.normalize() - df['scheduled_day'].dt.normalize()).dt.days

# appointment_weekday / appointment_month / appointment_year: needed to analyze scheduling patterns
# (e.g. which weekdays have worse attendance).
df['appointment_weekday'] = df['appointment_day'].dt.day_name()
df['appointment_month'] = df['appointment_day'].dt.month_name()
df['appointment_year'] = df['appointment_day'].dt.year

# age_group: bucket ages into clinically/operationally meaningful bands for reporting
# (matches the bands used in the Phase 2 Excel Pivot_Analysis tab, for consistency across tools).
def age_group(a):
    if a <= 12: return '0-12 Child'
    if a <= 18: return '13-18 Teen'
    if a <= 35: return '19-35 Young Adult'
    if a <= 50: return '36-50 Adult'
    if a <= 65: return '51-65 Senior'
    return '66+ Elderly'

df['age_group'] = df['age'].apply(age_group)

# waiting_time_group: buckets lead_time_days for easier grouped analysis / DAX measures later
def waiting_time_group(days):
    if days == 0: return 'Same day'
    if days <= 3: return '1-3 days'
    if days <= 7: return '4-7 days'
    if days <= 14: return '8-14 days'
    if days <= 30: return '15-30 days'
    return '31+ days'

df['waiting_time_group'] = df['lead_time_days'].apply(waiting_time_group)

# reminder_flag: renamed/clarified alias of sms_received for readability in downstream tools
df['reminder_flag'] = df['sms_received']

# scholarship_flag: renamed/clarified alias of scholarship (Bolsa Familia enrollment) - socio-economic proxy
df['scholarship_flag'] = df['scholarship']

# chronic_condition_count: how many of the three tracked chronic conditions a patient has.
# Useful as a single severity-style measure instead of three separate binary columns.
df['chronic_condition_count'] = df['hypertension'] + df['diabetes'] + df['alcoholism']

print("New columns added:")
print([c for c in df.columns if c not in ['patient_id','appointment_id','gender','scheduled_day','appointment_day',
      'age','neighbourhood','scholarship','hypertension','diabetes','alcoholism','handicap','sms_received','no_show']])


New columns added:
['no_show_flag', 'lead_time_days', 'appointment_weekday', 'appointment_month', 'appointment_year', 'age_group', 'waiting_time_group', 'reminder_flag', 'scholarship_flag', 'chronic_condition_count']


**Note on `appointment_period` (morning/afternoon/evening):** Phase 1 confirmed `appointment_day` has **no time
component** in this dataset (it's always midnight) — only `scheduled_day` carries a real timestamp, and that
reflects when the booking was made, not the time of the visit. Because the data doesn't actually support a
time-of-day-of-visit feature, we deliberately do **not** create `appointment_period`, per the project rule to
adapt rather than invent when a column isn't available.


In [12]:
df[['age','age_group','lead_time_days','waiting_time_group','appointment_weekday',
    'appointment_month','appointment_year','chronic_condition_count','no_show_flag']].head(10)


,age,age_group,lead_time_days,waiting_time_group,appointment_weekday,appointment_month,appointment_year,chronic_condition_count,no_show_flag
0,62,51-65 Senior,0,Same day,Friday,April,2016,1,0
1,56,51-65 Senior,0,Same day,Friday,April,2016,0,0
2,62,51-65 Senior,0,Same day,Friday,April,2016,0,0
3,8,0-12 Child,0,Same day,Friday,April,2016,0,0
4,56,51-65 Senior,0,Same day,Friday,April,2016,2,0
5,76,66+ Elderly,2,1-3 days,Friday,April,2016,1,0
6,23,19-35 Young Adult,2,1-3 days,Friday,April,2016,0,1
7,39,36-50 Adult,2,1-3 days,Friday,April,2016,0,1
8,21,19-35 Young Adult,0,Same day,Friday,April,2016,0,0
9,19,19-35 Young Adult,2,1-3 days,Friday,April,2016,0,0


## 3. Final Checks & Save

Quick sanity check on the cleaned dataset, then save it for use in Notebook 02 (EDA), the SQL phase, and Power BI.


In [13]:
print("Final shape:", df.shape)
print("\nDtypes:\n", df.dtypes)
print("\nAny remaining missing values:\n", df.isnull().sum().sum())
print("\nOverall no-show rate:", round(df['no_show_flag'].mean() * 100, 2), "%")


Final shape: (110521, 24)

Dtypes:
 patient_id                                 str
appointment_id                           int64
gender                                     str
scheduled_day              datetime64[us, UTC]
appointment_day            datetime64[us, UTC]
age                                      int64
neighbourhood                              str
scholarship                              int64
hypertension                             int64
diabetes                                 int64
alcoholism                               int64
handicap                                 int64
sms_received                             int64
no_show                                    str
no_show_flag                             int64
lead_time_days                           int64
appointment_weekday                        str
appointment_month                          str
appointment_year                         int32
age_group                                  str
waiting_time_group      


Any remaining missing values:
 0

Overall no-show rate: 20.19 %


In [14]:
OUT_PATH = '../data/cleaned/cleaned_appointments.csv'
df.to_csv(OUT_PATH, index=False)
print(f"Saved cleaned dataset to {OUT_PATH}")
print("Rows:", len(df), "| Columns:", len(df.columns))


Saved cleaned dataset to ../data/cleaned/cleaned_appointments.csv
Rows: 110521 | Columns: 24


## Summary

| Step | Rows Removed | Reason |
|---|---:|---|
| Invalid age (`age < 0`) | 1 | Logically invalid age value |
| Invalid lead time (negative) | 5 | Appointment logged as scheduled after the appointment date |
| **Total rows removed** | **6** | Out of 110,527 original rows (0.005%) |

**Columns added:** `no_show_flag`, `lead_time_days`, `appointment_weekday`, `appointment_month`,
`appointment_year`, `age_group`, `waiting_time_group`, `reminder_flag`, `scholarship_flag`,
`chronic_condition_count`.

**Columns deliberately NOT created:** `appointment_period` — the dataset does not contain a real appointment
time-of-day, so this would have been invented, not derived.

Next: **Notebook 02 — Exploratory Data Analysis**, using `data/cleaned/cleaned_appointments.csv` as the input.
